In [1]:
from src.MLP_grid_search import *

import torch
from torch import nn
import pandas as pd
import numpy as np

In [2]:
# select device to run this on
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
# create synthetic data

n_samples, n_features = 2000, 10                # number of samples & number of features

x = torch.randn(n_samples, n_features)          # randomly generated features
weights = torch.randn(n_features) * 5           # randomly generated weights

y = (x @ weights)                               # linear combinations of random features & weights
y += torch.randn(n_samples) * abs(y).mean()     # normal random error

y = nn.Sigmoid()(y).round()                     # sigmoid for binary classification

In [4]:
print(f"X Shape: {x.shape}\nY Shape: {y.shape}")

X Shape: torch.Size([2000, 10])
Y Shape: torch.Size([2000])


In [5]:
# hyperparameters to search are lists; a single value is held fixed
# hidden_dims are hidden-layer widths. So [64] is a 2-layer MLP and [64, 32, 16] is a 4-layer MLP.

param_dict = {

              # for hyperparameters to be tuned, enter list of values
              'lr': [1e-3, 5e-3, 1e-2],
              'activation': ['relu', 'tanh'],
              'hidden_dims': [[32], [64], [64, 32], [128, 64], [64, 32, 16]],
              'dropouts': [0.0, 0.1, 0.2, 0.5],
              'batch_size': [32, 64, 128, 256],

              # input single values for constant hyperparameters
              'num_epochs': 45,
              'wd': 1e-4,
              'last_act': 'sigmoid',
              'loss_fn': 'bce',
              'stop_early': False
              }

In [6]:
# run function to approximate best parameter combinations using tensor completion

best_estimated_params = return_best_k_params(hyperameter_dict = param_dict,              # dictionary of parameters
                                             X = x,                                      # features
                                             Y = y,                                      # target
                                             num_top_combinations = 3,                   # number of best estimated combinations to return
                                             tensor_entries = 16,                        # number of tensor entries to compute to infer the rest
                                             training_values = 500,                      # training entries for ML model
                                             testing_values = 500,                       # testing values for ML model
                                             num_tests = 3,                              # number of tests for gridsearch entries (uses average value)
                                             tensor_completion_model = 'cpd.smooth',     # tensor completion model
                                             task_type = 'classification',               # MLP task
                                             metric = 'accuracy',                        # metric for each gridsearch entry
                                             rank = 6,                                   # rank decomposition used for tensor completion
                                             device = device,                            # device
                                             verbose = True                              # prints intermediate progress if True
                                             )

16/480 total combinations in sparse tensor.

1/16 param_combinations done.
2/16 param_combinations done.
3/16 param_combinations done.
4/16 param_combinations done.
5/16 param_combinations done.
6/16 param_combinations done.
7/16 param_combinations done.
8/16 param_combinations done.
9/16 param_combinations done.
10/16 param_combinations done.
11/16 param_combinations done.
12/16 param_combinations done.
13/16 param_combinations done.
14/16 param_combinations done.
15/16 param_combinations done.
16/16 param_combinations done.

Running sparse tensor completion...
Done with sparse tensor completion!

Evaluating predicted best parameters.
Done!


In [7]:
for i in range(len(best_estimated_params)):
    best_estimated_params[i][0]['Evaluation'] = best_estimated_params[i][1]

hyperparams = list(best_estimated_params[0][0])

In [12]:
df = pd.DataFrame(np.array([[str(param_set[0][p]) for p in hyperparams] for param_set in best_estimated_params]),
             columns = hyperparams)
df['Evaluation'] = df['Evaluation'].astype('float').round(3)
df

,lr,activation,hidden_dims,dropouts,batch_size,Evaluation
0,0.005,relu,"[128, 64]",0.5,128,0.769
1,0.005,tanh,"[128, 64]",0.2,128,0.766
2,0.005,relu,"[128, 64]",0.2,128,0.757
